# p1 Ingestion service


## goal:
DICOM goes in, normalized pixel array + metadata row out.

1. pydicom.dcmread, option where/how to parse laterality
2. Extract possible metadata fields from dicoms, enable user to select which to keep eg. PatientID, StudyInstanceUID, StudyDate, PixelSpacing, BitsStored, TransferSyntaxUID, Manufacturer
3. storage format options? 16-bit lossless PNG or pkl? consider pro/cons
4. metadata write to psql

In [27]:
import pydicom
from pathlib import Path
import pandas as pd
from collections import defaultdict
from datetime import datetime

root = Path("/home/hassan/repo/2d-mg-workbench/unzipped/cancers/patient0097")

records = []

def parse_view_from_filename(path: Path):
    stem = path.stem.upper()
    laterality = "L" if "LEFT" in stem else "R" if "RIGHT" in stem else "UNKNOWN"
    view = "MLO" if "MLO" in stem else "CC" if "CC" in stem else "UNKNOWN"
    return laterality, view

for dcm_path in root.rglob("*.dcm"):
    ds = pydicom.dcmread(dcm_path)

    pid = str(ds.get("PatientID", "UNKNOWN"))

    arr = ds.pixel_array
    arr_lut = pydicom.pixel_data_handlers.apply_voi_lut(ds.pixel_array, ds)

    
    study_instance_UID = str(ds.get("StudyInstanceUID", ""))
    study_date = str(ds.get("StudyDate", ""))
    study_date = datetime.strptime(str(study_date), "%Y%m%d")

    transfer_syntax_UID = str(ds.get("TransferSyntaxUID", ""))
    photometric_interpretation = str(ds.get("PhotometricInterpretation", ""))
    w_center = str(ds.get("WindowCenter", ""))
    w_width = str(ds.get("WindowWidth", ""))

    img_lat = str(ds.get("ImageLaterality", ""))

    lat, view = parse_view_from_filename(dcm_path)


    records.append(
        {
            "patient_id": pid,
            "study_date": study_date,
            "file_name": dcm_path,
            'study_instance_UID' :study_instance_UID,
            "transfer_syntax_UID" :transfer_syntax_UID,
            'photometric_interpretation' : photometric_interpretation,
            'window_center' : w_center,
            'window_width' : w_width,
            "raw_array": arr,
            "lut_array": arr_lut,
            "image_laterality" : lat,
            "view_code" : view, 

        }
    )

    
df = pd.DataFrame(records)
df

,patient_id,study_date,file_name,study_instance_UID,transfer_syntax_UID,photometric_interpretation,window_center,window_width,raw_array,lut_array,image_laterality,view_code
0,0097,1995-11-14,/home/hassan/repo/2d-mg-workbench/unzipped/can...,1.2.276.0.7230010.3.1.4.1787169844.9554.145468...,,MONOCHROME2,128,256,"[[68, 72, 253, 253, 253, 253, 253, 253, 253, 2...","[[1092.0, 1156.235294117647, 4062.882352941176...",R,CC
1,0097,1995-11-14,/home/hassan/repo/2d-mg-workbench/unzipped/can...,1.2.276.0.7230010.3.1.4.1787169844.9554.145468...,,MONOCHROME2,128,256,"[[68, 72, 253, 253, 253, 253, 253, 253, 253, 2...","[[1092.0, 1156.235294117647, 4062.882352941176...",R,MLO
2,0097,1995-11-14,/home/hassan/repo/2d-mg-workbench/unzipped/can...,1.2.276.0.7230010.3.1.4.1787169844.9554.145468...,,MONOCHROME2,128,256,"[[254, 254, 254, 254, 254, 254, 254, 254, 254,...","[[4078.9411764705883, 4078.9411764705883, 4078...",L,MLO
3,0097,1995-11-14,/home/hassan/repo/2d-mg-workbench/unzipped/can...,1.2.276.0.7230010.3.1.4.1787169844.9554.145468...,,MONOCHROME2,126,253,"[[10, 9, 10, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[[170.62500000000009, 154.37499999999991, 170....",L,CC


In [26]:
def check_view_tag(root_path):
    mismatches = []
    for dcm_path in Path(root_path).rglob("*.dcm"):
        ds = pydicom.dcmread(dcm_path, stop_before_pixels=True)
        filename_view = "MLO" if "MLO" in dcm_path.name.upper() else "CC"
        tag_meaning = ""
        if "ViewCodeSequence" in ds and len(ds.ViewCodeSequence) > 0:
            tag_meaning = str(ds.ViewCodeSequence[0].get("CodeMeaning", ""))
        tag_view = "MLO" if "oblique" in tag_meaning.lower() else "CC" if "cranio" in tag_meaning.lower() else "UNKNOWN"
        if filename_view != tag_view:
            mismatches.append((dcm_path, filename_view, tag_view))
    return mismatches

mismatches = check_view_tag("/home/hassan/repo/2d-mg-workbench/unzipped")
print(f"{len(mismatches)} mismatches")

5182 mismatches
